# BSDE Primer: From Feynman-Kac to Numerical Solvers

A **Backward Stochastic Differential Equation (BSDE)** is a pair of adapted
processes $(Y, Z)$ satisfying

$$
Y_t = g(X_T) + \int_t^T f(X_s, Y_s, Z_s)\,ds - \int_t^T Z_s\,dW_s, \quad 0 \le t \le T.
$$

- $X_t$ — the **forward** (state) process, e.g. an Ornstein-Uhlenbeck diffusion.
- $f$ — the **driver** (generator / running cost).
- $g$ — the **terminal condition**.
- $Y_t$ — the **value** process; in the Markovian case $Y_t = u(t, X_t)$.
- $Z_t$ — the **control** process; $Z_t = \sigma(X_t)\,\partial_x u(t, X_t)$.

**Feynman-Kac** links BSDEs to quasilinear PDEs:

$$
\partial_t u + \tfrac{1}{2}\sigma^2 \partial_{xx} u + b\,\partial_x u
+ f(x, u, \sigma\,\partial_x u) = 0, \quad u(T, x) = g(x).
$$

This notebook illustrates the connection with a concrete linear example and
compares two numerical solvers.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

from ebsde.forward.ou_process import OrnsteinUhlenbeck
from ebsde.bsde.standard import StandardBSDE
from ebsde.solvers.pde import PDEBSDESolver
from ebsde.solvers.picard import PicardBSDESolver


## Linear BSDE: $f(x, y, z) = -y$, $g(x) = x^2$

We consider an Ornstein-Uhlenbeck forward process

$$
dX_t = -\kappa X_t\,dt + \sigma\,dW_t, \quad X_0 = x_0,
$$

with $\kappa = 1$, $\sigma = 1$, and the linear BSDE

$$
f(x, y, z) = -y, \quad g(x) = x^2, \quad T = 1.
$$

The Feynman-Kac solution is

$$
u(t, x) = e^{-(T-t)} \mathbb{E}[X_T^2 \mid X_t = x]
         = e^{-(T-t)} \bigl[\text{Var}(X_T|X_t) + (\mathbb{E}[X_T|X_t])^2\bigr].
$$

For OU:
$$
\mathbb{E}[X_T|X_t=x] = x\,e^{-\kappa(T-t)}, \quad
\text{Var}(X_T|X_t) = \frac{\sigma^2}{2\kappa}(1 - e^{-2\kappa(T-t)}).
$$


In [ ]:
import os
# --- Setup -----------------------------------------------------------
kappa, sigma, T = 1.0, 1.0, 1.0
ou = OrnsteinUhlenbeck(kappa=kappa, theta=0.0, sigma=sigma)

driver   = lambda x, y, z: -y
terminal = lambda x: x**2

bsde = StandardBSDE(forward=ou, driver=driver, terminal=terminal, T=T)

# --- PDE solver ------------------------------------------------------
pde_sol = PDEBSDESolver(bsde, n_x=200, n_t=200).solve()

x_grid = pde_sol['x_grid']
u0     = pde_sol['u'][0, :]          # u(0, x)

# --- Exact Feynman-Kac solution at t=0 --------------------------------
def exact_u0(x, kappa=kappa, sigma=sigma, T=T):
    tau = T
    mean_XT  = x * np.exp(-kappa * tau)
    var_XT   = (sigma**2 / (2*kappa)) * (1 - np.exp(-2*kappa*tau))
    E_XT2    = var_XT + mean_XT**2
    return np.exp(-tau) * E_XT2

u_exact = exact_u0(x_grid)

# --- Plot ---------------------------------------------------------------
os.makedirs('notebooks/figures', exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_grid, u_exact, 'k--', lw=2, label='Exact Feynman-Kac')
ax.plot(x_grid, u0,      'C0',  lw=2, label='PDEBSDESolver')
ax.set_xlabel('x')
ax.set_ylabel('u(0, x)')
ax.set_title('Linear BSDE: PDE solution vs exact at t=0')
ax.legend()
ax.set_xlim(-4, 4)
fig.tight_layout()
fig.savefig('notebooks/figures/01_pde_solution.png', dpi=120)
plt.close(fig)

print(f"Y0 (PDE, x=0) = {pde_sol['Y0']:.6f}")
print(f"Y0 (exact)    = {exact_u0(0.0):.6f}")
print("Figure saved → notebooks/figures/01_pde_solution.png")


## Three Solvers: PDE, Picard Iteration

The `ebsde` library provides two complementary solvers for standard BSDEs:

| Solver | Method | Best for |
|--------|--------|----------|
| `PDEBSDESolver` | Crank-Nicolson finite difference | Smooth, 1-D Markovian problems |
| `PicardBSDESolver` | Monte-Carlo + regression (Bouchard-Touzi) | Higher dimensions |

We now compare their $Y_0 = u(0, X_0)$ estimates for the same problem.


In [ ]:
import os
import numpy as np

# --- Picard solver -------------------------------------------------------
picard_sol = PicardBSDESolver(
    bsde, n_paths=10000, n_steps=50, n_picard=3
).solve()

# --- PDE solver (already run above) -------------------------------------
Y0_pde    = float(pde_sol['Y0'])
Y0_picard = float(picard_sol['Y0'])
Y0_exact  = float(exact_u0(0.0))


In [ ]:
print("=" * 50)
print(f"{'Solver':<20}  {'Y0':>10}  {'Error':>10}")
print("-" * 50)
print(f"{'Exact':.<20}  {Y0_exact:>10.6f}  {'—':>10}")
print(f"{'PDEBSDESolver':.<20}  {Y0_pde:>10.6f}  {abs(Y0_pde-Y0_exact):>10.2e}")
print(f"{'PicardBSDESolver':.<20}  {Y0_picard:>10.6f}  {abs(Y0_picard-Y0_exact):>10.2e}")
print("=" * 50)
print()
print("Picard convergence:", picard_sol['picard_convergence'])
